In [ ]:
import sqlite3, json
from pathlib import Path
import pandas as pd

# DATA_DIR is the campaign root; BATCH is the batch index of the clicked node.
# Both are injected by the robovast analysis runner (the search layout is flat:
# every batch's configs live directly under the campaign root, so campaign.db
# is what tells us which configs belong to this batch).
DATA_DIR = ''
BATCH = None

db = Path(DATA_DIR) / 'campaign.db'
assert db.exists(), f'no campaign.db under {DATA_DIR}'

con = sqlite3.connect(f'file:{db}?mode=ro', uri=True)
con.row_factory = sqlite3.Row
# Resolve this batch (fall back to the latest one if BATCH was not injected).
if BATCH is None:
    batch = con.execute('SELECT * FROM batch ORDER BY idx DESC LIMIT 1').fetchone()
else:
    batch = con.execute('SELECT * FROM batch WHERE idx = ?', (BATCH,)).fetchone()
assert batch is not None, f'batch {BATCH} not found'
units = con.execute('SELECT * FROM unit WHERE batch_id = ? ORDER BY id',
                    (batch['id'],)).fetchall()
print(f"batch idx={batch['idx']}: {len(units)} unit(s)")

In [ ]:
# One row per config evaluated in this batch: its objective(s), measures and params.
rows = []
for u in units:
    rec = {'config': u['config_name'], 'objective': u['objective'],
           'n_samples': u['n_samples'], 'status': u['status']}
    rec.update(json.loads(u['measures_json'] or '{}'))
    rec.update(json.loads(u['params_json'] or '{}'))
    rows.append(rec)
batch_df = pd.DataFrame(rows)
print(f'{len(batch_df)} config(s) in batch {batch["idx"]}')
batch_df

In [ ]:
import matplotlib.pyplot as plt

# Where this batch sampled the search space, coloured by objective.
if len(batch_df):
    obj = 'objective'
    measures = [c for c in ['max_tilt', 'drift_dist', 'landing_speed', 'control_effort']
                if c in batch_df.columns]
    if len(measures) >= 2:
        fig, ax = plt.subplots(figsize=(6, 5))
        sc = ax.scatter(batch_df[measures[0]], batch_df[measures[1]],
                        c=batch_df[obj], cmap='viridis')
        ax.set_xlabel(measures[0]); ax.set_ylabel(measures[1])
        fig.colorbar(sc, ax=ax, label=obj)
        ax.set_title(f'batch {batch["idx"]}: configs in behavior space')
        plt.tight_layout(); plt.show()
    display(batch_df.sort_values(obj, ascending=False).head(10))
else:
    print('No units recorded for this batch.')